# 🐦 BirdCLEF 2026 — Improved Pipeline
**EfficientNet-B2 + AMP + SpecAugment + ROC-AUC + TTA Ensemble**

In [ ]:
!pip install timm -q

In [ ]:
import numpy as np
import pandas as pd
import os
import gc
import librosa
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast   # AMP
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import timm

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
# =========================================================
# CONFIG  (tek yerden yönet)
# =========================================================

CFG = dict(
    # Audio
    SR          = 32000,
    DURATION    = 5,
    N_MELS      = 128,       # 128 → daha hızlı, 256 → daha detaylı
    N_FFT       = 1024,
    HOP_LENGTH  = 512,
    FMIN        = 20,
    FMAX        = 16000,

    # Training
    BATCH_SIZE  = 32,
    EPOCHS      = 20,
    FOLDS       = 5,
    LR          = 1e-3,
    WEIGHT_DECAY= 1e-4,
    LABEL_SMOOTH= 0.05,
    WARMUP_EPOCHS=2,
    NUM_WORKERS = 4,
    SEED        = 42,

    # SpecAugment
    FREQ_MASK   = 30,
    TIME_MASK   = 40,

    # Paths
    DATA_ROOT   = "/kaggle/input/birdclef-2026",
)

CFG['SAMPLES']   = CFG['SR'] * CFG['DURATION']
CFG['TRAIN_CSV'] = f"{CFG['DATA_ROOT']}/train.csv"
CFG['TRAIN_DIR'] = f"{CFG['DATA_ROOT']}/train_audio"
CFG['TEST_DIR']  = f"{CFG['DATA_ROOT']}/test_soundscapes"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

torch.manual_seed(CFG['SEED'])
np.random.seed(CFG['SEED'])

In [ ]:
# =========================================================
# DATA
# =========================================================

train_df = pd.read_csv(CFG['TRAIN_CSV'])

labels   = sorted(train_df.primary_label.unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
NUM_CLASSES = len(labels)

train_df['target'] = train_df.primary_label.map(label2id)

print(f"Samples: {len(train_df)} | Classes: {NUM_CLASSES}")
print(f"Label distribution (top 10):\n{train_df.primary_label.value_counts().head(10)}")
train_df.head()

In [ ]:
# =========================================================
# AUDIO → MEL  (tek, merkezi fonksiyon)
# =========================================================

def audio_to_mel(path_or_array, sr=None, start_sample=0):
    """
    Dosya yolu veya numpy dizisinden normalize edilmiş mel-spectrogram döner.
    Returns: np.float32 shape (N_MELS, time_frames)
    """
    if isinstance(path_or_array, (str, bytes, os.PathLike)):
        audio, _ = librosa.load(path_or_array, sr=CFG['SR'], mono=True)
    else:
        audio = path_or_array

    # Belirli bir başlangıç noktasından SAMPLES kadar al
    audio = audio[start_sample : start_sample + CFG['SAMPLES']]

    # Pad veya kırp
    if len(audio) < CFG['SAMPLES']:
        audio = np.pad(audio, (0, CFG['SAMPLES'] - len(audio)))
    else:
        audio = audio[:CFG['SAMPLES']]

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=CFG['SR'],
        n_mels=CFG['N_MELS'],
        n_fft=CFG['N_FFT'],
        hop_length=CFG['HOP_LENGTH'],
        fmin=CFG['FMIN'],
        fmax=CFG['FMAX'],
    )
    mel = librosa.power_to_db(mel, ref=np.max).astype(np.float32)

    # Instance normalization
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)

    return mel   # (N_MELS, time_frames)

In [ ]:
# =========================================================
# SPECAUGMENT
# =========================================================

def spec_augment(mel, freq_mask=CFG['FREQ_MASK'], time_mask=CFG['TIME_MASK']):
    """Frequency & time masking (SpecAugment)."""
    mel = mel.copy()
    _, T = mel.shape

    # Frequency mask
    f0 = np.random.randint(0, freq_mask)
    f  = np.random.randint(0, CFG['N_MELS'] - f0)
    mel[f : f + f0, :] = 0

    # Time mask
    t0 = np.random.randint(0, time_mask)
    t  = np.random.randint(0, T - t0)
    mel[:, t : t + t0] = 0

    return mel

In [ ]:
# =========================================================
# DATASET
# =========================================================

class BirdDataset(Dataset):
    def __init__(self, df, audio_dir, is_train=True):
        self.df        = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.is_train  = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(self.audio_dir, row['filename'])

        # Eğitimde rastgele başlangıç noktası → veri çeşitliliği
        if self.is_train:
            try:
                duration = librosa.get_duration(path=path)
                max_start = max(0, int(duration * CFG['SR']) - CFG['SAMPLES'])
                start = np.random.randint(0, max_start + 1) if max_start > 0 else 0
            except Exception:
                start = 0
        else:
            start = 0

        mel    = audio_to_mel(path, start_sample=start)

        if self.is_train:
            mel = spec_augment(mel)

        tensor = torch.from_numpy(mel).unsqueeze(0)   # (1, N_MELS, T)

        if self.is_train:
            return tensor, torch.tensor(row['target'], dtype=torch.long)
        return tensor

In [ ]:
# =========================================================
# MODEL
# =========================================================

class BirdModel(nn.Module):
    def __init__(self, num_classes, backbone='tf_efficientnet_b2'):
        super().__init__()
        self.backbone = timm.create_model(
            backbone,
            pretrained=True,
            in_chans=1,
            num_classes=0,     # features only
        )
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.GELU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

In [ ]:
# =========================================================
# MIXUP
# =========================================================

def mixup(x, y, alpha=0.4):
    """Mixup data augmentation. Returns mixed x and two targets + lam."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    idx   = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    return x_mix, y, y[idx], lam


def mixup_criterion(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)

In [ ]:
# =========================================================
# TRAIN / VALID  (AMP + Mixup + Label Smoothing)
# =========================================================

def train_one_epoch(loader, model, optimizer, loss_fn, scaler):
    model.train()
    total_loss = 0.0

    for mel, target in tqdm(loader, desc='  train', leave=False):
        mel, target = mel.to(DEVICE), target.to(DEVICE)

        # Mixup
        mel, target_a, target_b, lam = mixup(mel, target)

        optimizer.zero_grad()
        with autocast():
            logits = model(mel)
            loss   = mixup_criterion(loss_fn, logits, target_a, target_b, lam)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def validate(loader, model, loss_fn):
    model.eval()
    total_loss = 0.0
    all_probs, all_targets = [], []

    for mel, target in tqdm(loader, desc='  valid', leave=False):
        mel, target = mel.to(DEVICE), target.to(DEVICE)
        with autocast():
            logits = model(mel)
            loss   = loss_fn(logits, target)

        total_loss += loss.item()
        all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
        all_targets.append(target.cpu().numpy())

    all_probs   = np.concatenate(all_probs,   axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    # ROC-AUC (macro) — BirdCLEF'in asıl değerlendirme metriği
    try:
        auc = roc_auc_score(
            np.eye(NUM_CLASSES)[all_targets],
            all_probs,
            average='macro',
            multi_class='ovr'
        )
    except ValueError:
        auc = 0.0

    return total_loss / len(loader), auc

In [ ]:
# =========================================================
# CROSS-VALIDATION TRAIN LOOP
# =========================================================

skf    = StratifiedKFold(n_splits=CFG['FOLDS'], shuffle=True, random_state=CFG['SEED'])
models = []
oof_auc_list = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(train_df, train_df['target'])):
    print(f"\n{'='*55}")
    print(f"  FOLD {fold + 1} / {CFG['FOLDS']}")
    print(f"{'='*55}")

    tr_df  = train_df.iloc[tr_idx]
    val_df = train_df.iloc[val_idx]

    train_ds     = BirdDataset(tr_df,  audio_dir=CFG['TRAIN_DIR'], is_train=True)
    valid_ds     = BirdDataset(val_df, audio_dir=CFG['TRAIN_DIR'], is_train=False)
    train_loader = DataLoader(train_ds, batch_size=CFG['BATCH_SIZE'], shuffle=True,
                              num_workers=CFG['NUM_WORKERS'], pin_memory=True, drop_last=True)
    valid_loader = DataLoader(valid_ds, batch_size=CFG['BATCH_SIZE'], shuffle=False,
                              num_workers=CFG['NUM_WORKERS'], pin_memory=True)

    model     = BirdModel(num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['LR'], weight_decay=CFG['WEIGHT_DECAY'])
    loss_fn   = nn.CrossEntropyLoss(label_smoothing=CFG['LABEL_SMOOTH'])
    scaler    = GradScaler()

    # Warmup + CosineAnnealing
    scheduler_warmup = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, total_iters=CFG['WARMUP_EPOCHS'])
    scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG['EPOCHS'] - CFG['WARMUP_EPOCHS'], eta_min=1e-6)
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[scheduler_warmup, scheduler_cosine],
        milestones=[CFG['WARMUP_EPOCHS']])

    best_auc  = 0.0
    best_path = f"best_fold{fold}.pth"

    for epoch in range(CFG['EPOCHS']):
        tr_loss           = train_one_epoch(train_loader, model, optimizer, loss_fn, scaler)
        val_loss, val_auc = validate(valid_loader, model, loss_fn)
        scheduler.step()

        flag = '  ← best' if val_auc > best_auc else ''
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), best_path)

        lr_now = scheduler.get_last_lr()[0]
        print(f"  Ep {epoch+1:2d}/{CFG['EPOCHS']} | "
              f"Train: {tr_loss:.4f} | Val: {val_loss:.4f} | "
              f"AUC: {val_auc:.4f} | LR: {lr_now:.2e}{flag}")

    print(f"  ✅ Best AUC: {best_auc:.4f}")
    oof_auc_list.append(best_auc)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    models.append(model)

    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*55}")
print(f"  CV Mean AUC: {np.mean(oof_auc_list):.4f} ± {np.std(oof_auc_list):.4f}")
print(f"{'='*55}")

In [ ]:
# =========================================================
# TEST INFERENCE  (sliding window TTA)
# =========================================================

@torch.no_grad()
def predict_file(path):
    """Ses dosyasından sliding window + flip TTA ile tahmin üretir."""
    audio, _ = librosa.load(path, sr=CFG['SR'], mono=True)

    STEP   = CFG['SAMPLES'] // 2
    starts = list(range(0, max(1, len(audio) - CFG['SAMPLES'] + 1), STEP))
    chunk_preds = []

    for start in starts:
        chunk = audio[start : start + CFG['SAMPLES']]
        mel   = audio_to_mel(chunk)   # merkezi fonksiyon

        t      = torch.tensor(mel).unsqueeze(0).unsqueeze(0).to(DEVICE)  # (1,1,H,W)
        t_flip = torch.flip(t, dims=[-1])                                 # time-flip TTA

        p = torch.zeros(NUM_CLASSES, device=DEVICE)
        for m in models:
            m.eval()
            with autocast():
                p += torch.softmax(m(t),      dim=1)[0]
                p += torch.softmax(m(t_flip), dim=1)[0]
        p /= len(models) * 2
        chunk_preds.append(p.cpu().numpy())

    return np.mean(chunk_preds, axis=0)

In [ ]:
# =========================================================
# SUBMISSION
# =========================================================

test_df     = pd.read_csv(f"{CFG['DATA_ROOT']}/sample_submission.csv")
predictions = []

for row in tqdm(test_df.itertuples(), total=len(test_df), desc='Inference'):
    file_id = getattr(row, 'audio_id', getattr(row, 'row_id', None))
    path    = os.path.join(CFG['TEST_DIR'], f"{file_id}.ogg")
    predictions.append(predict_file(path))

predictions = np.array(predictions)

submission  = pd.DataFrame(predictions, columns=labels)
submission.insert(0, 'row_id', test_df['row_id'])
submission.to_csv('submission.csv', index=False)
print('✅ submission.csv kaydedildi!')
submission.head()

In [ ]:
# =========================================================
# MODEL EXPORT  (HuggingFace için)
# =========================================================
import json

# label2id / id2label kaydet
with open('label2id.json', 'w') as f:
    json.dump(label2id, f)
with open('id2label.json', 'w') as f:
    json.dump(id2label, f)

# Config kaydet
with open('config.json', 'w') as f:
    json.dump({k: v for k, v in CFG.items() if isinstance(v, (int, float, str))}, f, indent=2)

# En iyi fold modelini ensemble ağırlığı olarak kaydet
# (ya da tüm fold'ları fold_0.pth ... fold_4.pth şeklinde HF'e yükle)
best_fold_idx = int(np.argmax(oof_auc_list))
import shutil
shutil.copy(f'best_fold{best_fold_idx}.pth', 'model.pth')

print(f'✅ En iyi fold: {best_fold_idx} (AUC={oof_auc_list[best_fold_idx]:.4f})')
print('✅ model.pth, label2id.json, id2label.json, config.json kaydedildi.')